In [1]:
# ================================================================
# TORCH: Factory-Level Risk Index — Myanmar Apparel Supply Chains
# Stage 3: Feature Extraction — Topic Modeling + NER
# ================================================================
# This stage has two goals:
#   1. Topic Modeling (LDA) — uncover latent risk themes across
#      the article corpus without any predefined labels
#   2. Named Entity Recognition (NER) — extract factory names
#      from article text and link articles to known factories

import pandas as pd
import ast
import spacy
import re
from gensim import corpora
from gensim.models import LdaModel
from thefuzz import process

# pip install gensim spacy
# python -m spacy download en_core_web_sm

news  = pd.read_csv("stage2_news_preprocessed.csv")
osh   = pd.read_csv("stage1_osh.csv")
bhrrc = pd.read_csv("stage1_bhrrc_labels.csv")

# tokens were saved as strings — convert back to lists
news["tokens"] = news["tokens"].apply(ast.literal_eval)

print("Articles loaded:", len(news))

# ================================================================
# PART A — TOPIC MODELING (LDA)
# ================================================================
# Latent Dirichlet Allocation (LDA) is an unsupervised method that
# discovers hidden thematic structure in a collection of documents.
# Each topic is a probability distribution over words, and each
# document is a mixture of topics. We use this to validate that
# our corpus naturally contains the risk themes we expect
# (e.g. overtime, wages, safety) without imposing them manually.

# ----------------------------------------------------------------
# A1. Build dictionary and corpus for LDA
#     - Dictionary: maps each unique token to an integer ID
#     - Corpus: represents each article as a bag-of-words vector
#       (list of (token_id, frequency) pairs)
# ----------------------------------------------------------------

dictionary = corpora.Dictionary(news["tokens"])

# Filter out tokens that appear in fewer than 5 articles (too rare)
# or more than 50% of articles (too common to be informative)
dictionary.filter_extremes(no_below=5, no_above=0.5)

corpus = [dictionary.doc2bow(tokens) for tokens in news["tokens"]]

print("Dictionary size after filtering:", len(dictionary))
print("Corpus size:", len(corpus))

# ----------------------------------------------------------------
# A2. Train LDA model
#     num_topics: we use 8 to align with the 8 Better Work CAT
#     clusters (ILO & IFC, 2025), giving us one topic per
#     expected risk dimension
#     passes: number of training iterations over the corpus
#     random_state: set for reproducibility
# ----------------------------------------------------------------

NUM_TOPICS = 8

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=NUM_TOPICS,
    passes=15,
    random_state=42
)

print("\nLDA Topics (top 10 words per topic):")
for i, topic in lda_model.print_topics(num_topics=NUM_TOPICS, num_words=10):
    print(f"  Topic {i}: {topic}")

# ----------------------------------------------------------------
# A3. Assign dominant topic to each article
#     For each article, get the topic with the highest probability
# ----------------------------------------------------------------

def get_dominant_topic(bow):
    topic_probs = lda_model.get_document_topics(bow)
    if not topic_probs:
        return None, 0.0
    dominant = max(topic_probs, key=lambda x: x[1])
    return dominant[0], round(dominant[1], 4)

news["dominant_topic"], news["topic_prob"] = zip(
    *[get_dominant_topic(bow) for bow in corpus]
)

print("\nArticle count per dominant topic:")
print(news["dominant_topic"].value_counts().sort_index())

# ================================================================
# PART B — NAMED ENTITY RECOGNITION (NER)
# ================================================================
# We use spaCy's NER to extract organisation names (ORG entities)
# from article text. These are candidate factory names mentioned
# in the complaint narratives. We then fuzzy-match them against
# the OSH factory registry to link articles to known factories.

nlp = spacy.load("en_core_web_sm")

# ----------------------------------------------------------------
# B1. Extract ORG entities from each article's clean content
# ----------------------------------------------------------------

def extract_orgs(text):
    if pd.isna(text) or len(str(text)) == 0:
        return []
    doc = nlp(str(text)[:50000])  # spaCy has a token limit safeguard
    orgs = list(set([
        ent.text.strip()
        for ent in doc.ents
        if ent.label_ == "ORG" and len(ent.text.strip()) > 3
    ]))
    return orgs

print("\nRunning NER on articles (this may take a few minutes)...")
news["extracted_orgs"] = news["content_clean"].apply(extract_orgs)

# Quick check — how many articles had at least one ORG extracted
has_org = (news["extracted_orgs"].apply(len) > 0).sum()
print(f"Articles with at least one ORG entity: {has_org}/{len(news)}")

# ----------------------------------------------------------------
# B2. Fuzzy match extracted ORGs to OSH factory registry
#     For each article, try to match any extracted org name
#     to a known factory in the OSH registry.
#     We take the best match across all extracted orgs.
# ----------------------------------------------------------------

FUZZY_THRESHOLD = 90
osh_names = osh["name_lower"].tolist()

def match_orgs_to_osh(orgs):
    best_os_id, best_name, best_score = None, None, 0
    for org in orgs:
        result = process.extractOne(
            org.lower().strip(), osh_names, score_cutoff=FUZZY_THRESHOLD
        )
        if result:
            name, score = result
            if score > best_score:
                best_score = score
                idx = osh_names.index(name)
                best_os_id = osh.iloc[idx]["os_id"]
                best_name  = osh.iloc[idx]["name"]
    return best_os_id, best_name, best_score if best_score > 0 else None

print("Matching extracted ORGs to OSH registry...")
news[["os_id", "osh_factory_matched", "ner_match_score"]] = news["extracted_orgs"].apply(
    lambda orgs: pd.Series(match_orgs_to_osh(orgs))
)

matched   = news["os_id"].notna().sum()
unmatched = news["os_id"].isna().sum()
print(f"Articles matched to OSH factory: {matched}")
print(f"Articles unmatched: {unmatched}")

print("\nSample NER matches:")
print(
    news[news["os_id"].notna()][
        ["title", "osh_factory_matched", "ner_match_score"]
    ].head(8).to_string(index=False)
)

# ----------------------------------------------------------------
# B3. Also bring in BHRRC factory matches via title keyword search
#     Some articles explicitly mention a factory name in the title.
#     We fuzzy match the title against BHRRC factory names as a
#     secondary linking method for articles NER missed.
# ----------------------------------------------------------------

bhrrc_names = bhrrc["factory_name"].str.lower().str.strip().tolist()

def match_title_to_bhrrc(title):
    result = process.extractOne(
        str(title).lower().strip(), bhrrc_names, score_cutoff=FUZZY_THRESHOLD
    )
    if result:
        name, score = result
        idx = bhrrc_names.index(name)
        return bhrrc.iloc[idx]["factory_name"], score
    return None, None

# Only apply to articles that NER couldn't match
unmatched_mask = news["os_id"].isna()
news.loc[unmatched_mask, ["bhrrc_factory_matched", "bhrrc_match_score"]] = (
    news.loc[unmatched_mask, "title"]
    .apply(lambda t: pd.Series(match_title_to_bhrrc(t)))
    .values
)

bhrrc_matched = news["bhrrc_factory_matched"].notna().sum()
print(f"\nAdditional articles matched via title→BHRRC: {bhrrc_matched}")

# ================================================================
# PART C — Save outputs for Stage 4
# ================================================================

news.to_csv("stage3_news_features.csv", index=False, encoding="utf-8-sig")
lda_model.save("lda_model")
dictionary.save("lda_dictionary")

print("\nStage 3 done.")
print("  stage3_news_features.csv — articles with topic + NER features")
print("  lda_model / lda_dictionary — saved for reuse in reporting")

ModuleNotFoundError: No module named 'spacy'